# PlantVision 2.1 — houseplant disease pipeline (clean)

This is a single, end-to-end, **deterministic** training notebook that
produces every artefact `server/` reads at runtime:

```
models/
├── classifier.pth              ← KD student weights (MobileNetV3-Large)
├── classifier.onnx             ← FP32 export (sanity / fallback)
├── classifier_int8.onnx        ← static INT8 QDQ (production)
├── classes.json                ← {class_names, training.img_size, ood, kd, ...}
├── temperature.json            ← {temperature: T}
├── conformal.json              ← RAPS calibration {q_hat, alpha, ...}
├── detector.{pt,onnx}          ← Stage 1 (only re-trained if a YOLO dataset is supplied)
├── training_metrics.json
└── model_card.json
```

## Class set — 7 trainable classes (v2.1)

```
bacterial_leaf_spot, gray_mold_botrytis, healthy, leaf_spot_fungal,
not_a_plant, powdery_mildew, viral_disease
```

The four physiological conditions from the v2.0 design — `edema`,
`leaf_scorch_underwatering`, `nutrient_deficiency`, `sunburn_or_scorch` —
are **deferred until labelled real-world recall data is added**. Public
sources have very few clean indoor examples for these and including
them with proxy labels collapses recall on the diseases we already
have data for. Adding them later is a config change in §1, not a
re-architecture.

## Why these decisions

* **Loss-side balancing only.** Logit Adjustment (Menon ICLR 2021,
  τ=0.7) is used; the `WeightedRandomSampler` is **off** by default to
  avoid double-correction on rare classes. A flag is exposed but the
  default is what we ship.
* **Three independent val slices, one ship gate.**
  `val_realworld` = primary disease macro-F1 gate (≥ 0.70, no class
  with recall=0, healthy-bias < 1.5). `val_indomain` drives early
  stopping. `val_studio` (PlantVillage) is a transfer/background-bias
  monitor — never blocking.
* **Two dedicated calibration splits**, both held out from the train
  set: `temperature_cal` (~5 %) for INT8 calibration + temperature
  scaling, `conformal_cal` (~10 %) for RAPS quantile fitting. Both are
  restricted to the deployment distribution (`plantdoc + inat`),
  never PlantVillage, so conformal exchangeability holds at deploy.
* **EMA student** during KD (decay=0.999) — stabilises the student in
  the small-data regime. The EMA copy is what we export.
* **Static INT8 (QDQ) ships only if `val_realworld` disease F1 drops
  by < 0.02** vs FP32. Larger drops fail the cell loudly.
* **Fail-loud everywhere.** Every block raises `RuntimeError` on
  silent-failure paths (empty splits, taxonomy drift, label leakage,
  zero-recall classes, INT8 drift > 0.05).

Total training time on a single A100-40GB is ≈ 50–70 min (data fetch
included; ≈ 25 min on Colab T4 with smaller batch).


## §1 — Setup: deps, schema, taxonomy, eval helpers

This cell is idempotent — re-running it does not redo anything besides
imports and folder creation. It also:

* installs versioned deps,
* sets every seed,
* defines the `CropAnnotation` sidecar schema,
* declares the canonical `CLASSES` list (the alphabetical order is
  what `ImageFolder` will produce at training time and what every
  downstream artefact expects).


In [ ]:
# §1 — Setup
import os, sys, subprocess, random, json, hashlib, pathlib, re, shutil, time, zipfile, math
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Optional

# Pinned to versions that are known to import together; if your env
# already has matching versions this is a no-op.
# NOTE: numpy is intentionally unpinned. Recent torch / torchvision
# wheels (Colab default) link against numpy 2.x ABI; forcing
# numpy<2 breaks `from torchvision import …` with a binary
# incompatibility ValueError.
PIP_PKGS = [
    'torch>=2.2', 'torchvision', 'timm>=1.0',
    'ultralytics>=8.2', 'onnx>=1.16', 'onnxruntime>=1.17',
    'imagehash', 'Pillow', 'scikit-learn>=1.4', 'pandas',
    'opencv-python-headless>=4.9', 'tqdm', 'matplotlib',
    'requests>=2.32', 'datasets>=2.18', 'PyYAML',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PIP_PKGS])

import numpy as np, requests, torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms as T
from sklearn.metrics import f1_score, recall_score, confusion_matrix
from tqdm import tqdm

# ── Reproducibility ─────────────────────────────────────────────
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True       # faster training
torch.backends.cudnn.deterministic = False  # honest about the trade-off

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device={device}  torch={torch.__version__}')

# ── Sidecar metadata schema ─────────────────────────────────────
@dataclass
class CropAnnotation:
    plant_id: str
    species: Optional[str]
    diagnosis: str
    severity: str               # low | medium | high | unknown
    leaf_or_whole_plant: str    # leaf | whole_plant
    capture_session: str
    source: str                 # inat | coco | plantvillage | plantdoc
    label_confidence: float     # in [0, 1]
    bbox: List[int]             # [x, y, w, h]; [0,0,0,0] for whole-image

def write_sidecar(p: Path, ann: CropAnnotation):
    p.with_suffix('.json').write_text(json.dumps(asdict(ann), ensure_ascii=False, indent=2))
def read_sidecar(p: Path):
    s = p.with_suffix('.json'); return json.loads(s.read_text()) if s.exists() else None

# ── Folder layout ───────────────────────────────────────────────
ROOT          = pathlib.Path('data_v2')
CLS_ROOT      = ROOT / 'classification'
TRAIN_ROOT    = CLS_ROOT / 'train'
VAL_ROOT      = CLS_ROOT / 'val'
VAL_INDOMAIN  = VAL_ROOT / 'indomain'
VAL_PRIMARY   = VAL_ROOT / 'realworld'              # ship gate
VAL_STUDIO    = VAL_ROOT / 'plantvillage_studio'    # transfer monitor
TEMP_CAL      = VAL_ROOT / 'temperature_cal'        # INT8 + temperature
CONFORMAL_CAL = VAL_ROOT / 'conformal_cal'          # RAPS

# v2.1 — 7 trainable classes. The four physiological conditions are
# deferred until labelled real-world data is added (see §0).
CLASSES = [
    'bacterial_leaf_spot', 'gray_mold_botrytis', 'healthy',
    'leaf_spot_fungal', 'not_a_plant', 'powdery_mildew', 'viral_disease',
]
SPLIT_ROOTS = (VAL_INDOMAIN, VAL_PRIMARY, VAL_STUDIO, TEMP_CAL, CONFORMAL_CAL)

def ensure_layout():
    for c in CLASSES:
        (TRAIN_ROOT / c).mkdir(parents=True, exist_ok=True)
    for d in SPLIT_ROOTS:
        d.mkdir(parents=True, exist_ok=True)
ensure_layout()

# ── Eval helpers (used by every later phase) ────────────────────
NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
EVAL_WORKERS = min(4, os.cpu_count() or 2)
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def iter_images(root: Path):
    for p in Path(root).rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            yield p

def count_images(root):
    return sum(1 for _ in iter_images(root)) if Path(root).exists() else 0

def _seed_worker(worker_id):
    s = (SEED + worker_id) % (2**32)
    np.random.seed(s); random.seed(s)

def _loader_kwargs(workers, **extra):
    kw = dict(num_workers=workers,
              pin_memory=(device == 'cuda'),
              worker_init_fn=_seed_worker, **extra)
    if workers > 0:
        kw['persistent_workers'] = True
        kw['prefetch_factor'] = 2
    return kw

def make_eval_loader(root, img_size, batch=128, workers=EVAL_WORKERS, class_to_idx=None):
    """Build a deterministic eval loader whose label space matches training.

    `ImageFolder` indexes classes by sorted directory listing — an eval
    slice that is missing one trained class would silently shift the
    label IDs. `class_to_idx` is the train-time map; we remap `target`
    so y_true is comparable to model `argmax` on every slice.
    """
    tx = T.Compose([T.Resize(int(img_size * 1.05)),
                    T.CenterCrop(img_size), T.ToTensor(), NORM])
    target_transform = None
    if class_to_idx is not None:
        tmp = ImageFolder(str(root))
        unknown = [c for c in tmp.classes if c not in class_to_idx]
        if unknown:
            raise RuntimeError(
                f'Eval folder {root} has classes outside training taxonomy: {unknown}.'
            )
        slice_to_train = {tmp.class_to_idx[c]: class_to_idx[c] for c in tmp.classes}
        target_transform = slice_to_train.__getitem__
    ds = ImageFolder(str(root), transform=tx, target_transform=target_transform)
    if len(ds) == 0:
        raise RuntimeError(f'Eval folder is empty: {root}.  Run §2 first.')
    dl = DataLoader(ds, batch_size=batch, shuffle=False, **_loader_kwargs(workers))
    return ds, dl

@torch.inference_mode()
def evaluate(model, dl, class_names, device, name=''):
    """Disease macro-F1 + per-class recall + confusion matrix + healthy-bias."""
    model.eval()
    y_true, y_pred = [], []
    for x, y in dl:
        logits = model(x.to(device, non_blocking=True))
        y_pred.extend(logits.argmax(1).detach().cpu().tolist())
        y_true.extend(y.tolist())
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    disease_idx = [i for i, n in enumerate(class_names) if n not in ('healthy', 'not_a_plant')]
    f1 = float(f1_score(y_true, y_pred, labels=disease_idx, average='macro', zero_division=0))
    recall = recall_score(y_true, y_pred, labels=list(range(len(class_names))),
                          average=None, zero_division=0)
    h = class_names.index('healthy') if 'healthy' in class_names else None
    bias = float((y_pred == h).mean() / max((y_true == h).mean(), 1e-9)) if h is not None else 0.0
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names)))).tolist()
    print(f'[{name}] disease_macro_f1={f1:.3f}  healthy_bias={bias:.2f}  n={len(y_true)}')
    if 'val_realworld' in name or 'final' in name:
        for c, r in zip(class_names, recall):
            print(f'    recall[{c:25s}] = {r:.3f}')
    return {'name': name, 'disease_macro_f1': f1, 'healthy_bias': bias,
            'per_class_recall': {c: float(r) for c, r in zip(class_names, recall)},
            'confusion_matrix': cm}

# Persisted run log — every epoch appends a row. Useful when the colab
# session dies mid-train and you want to compare final to half-trained.
RUN_LOG = Path('models/run_log.jsonl')
RUN_LOG.parent.mkdir(parents=True, exist_ok=True)
def log_run(row: dict):
    row = {'ts': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), **row}
    with RUN_LOG.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')


## §2 — Data acquisition + group-aware splits

Public sources only. No manual upload. Every saved image gets a
sidecar JSON so the splitter can be group-aware (`(source, plant_id)`
goes together, never split across train and a held-out set).

| Source | Role |
|---|---|
| **PlantDoc** (HF) | real-world train + `val_realworld` seed |
| **PlantVillage** (HF) | capped train + `val_studio`. PlantVillage **never** enters `val_realworld`, `temp_cal` or `conformal_cal` (would break exchangeability) |
| **iNaturalist** | healthy houseplant leaves (Ficus / Monstera / …) and a small Botrytis supplement, **`Green Leaves` annotation only** so off-host imagery (fruit rot, agar plates) cannot leak |
| **COCO val2017** | `not_a_plant` images, plant-like categories filtered out |

Set `REBUILD_GENERATED_SPLITS = False` if you only want to top-up an
existing run.


In [ ]:
# §2 — Data acquisition + group-aware splits
from io import BytesIO
from PIL import Image as PILImage

UA = {'User-Agent': 'PlantVision/2.1 (research; contact: local-notebook)'}
REBUILD_GENERATED_SPLITS = True

if REBUILD_GENERATED_SPLITS:
    for d in (TRAIN_ROOT, VAL_ROOT):
        if d.exists():
            shutil.rmtree(d)
    ensure_layout()

# ── HTTP helpers (with backoff) ──────────────────────────────────
def http_get_json(url, params=None, timeout=30, retries=4):
    last = None; delay = 0.8
    for _ in range(retries):
        try:
            r = requests.get(url, params=params, headers=UA, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                last = requests.HTTPError(f'{r.status_code} {r.reason} on {url}')
                time.sleep(delay); delay *= 2; continue
            r.raise_for_status(); return r.json()
        except Exception as e:
            last = e; time.sleep(delay); delay *= 2
    raise last if last is not None else RuntimeError(f'http_get_json exhausted retries: {url}')

def http_download(url, dest, timeout=120, retries=4):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + '.tmp')
    last = None; delay = 1.0
    for _ in range(retries):
        try:
            with requests.get(url, headers=UA, stream=True, timeout=timeout) as r:
                if r.status_code in (429, 500, 502, 503, 504):
                    last = requests.HTTPError(f'{r.status_code} {r.reason} on {url}')
                    time.sleep(delay); delay *= 2; continue
                r.raise_for_status()
                with open(tmp, 'wb') as f:
                    for chunk in r.iter_content(1 << 20):
                        if chunk: f.write(chunk)
            tmp.replace(dest); return dest
        except Exception as e:
            last = e
            if tmp.exists():
                try: tmp.unlink()
                except Exception: pass
            time.sleep(delay); delay *= 2
    raise last if last is not None else RuntimeError(f'http_download exhausted retries: {url}')

def http_get_bytes(url, timeout=20, retries=4):
    delay = 0.5
    for _ in range(retries):
        try:
            r = requests.get(url, headers=UA, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(delay); delay *= 2; continue
            r.raise_for_status(); return r.content
        except Exception:
            time.sleep(delay); delay *= 2
    return None  # callers treat None as a soft skip (per-photo iNat)

def _stable_frac(text: str) -> float:
    return int(hashlib.sha1(text.encode('utf-8')).hexdigest()[:12], 16) / float(16**12 - 1)

def _save_rgb(img, out: Path, quality=92):
    out.parent.mkdir(parents=True, exist_ok=True)
    img.convert('RGB').save(out, quality=quality)

def _valid_image_bytes(body, min_side=160):
    try:
        im = PILImage.open(BytesIO(body)).convert('RGB')
        if min(im.size) < min_side: return None
        return im
    except Exception:
        return None

# ── iNaturalist (Leaves annotation, research-grade only) ────────
INAT_BASE = 'https://api.inaturalist.org/v1'
INAT_LEAF_TERM_ID, INAT_LEAF_TERM_VALUE_ID = 36, 38   # iNat 'Leaves' annotation, value='Green Leaves'

def inat_resolve_taxon(name, rank='genus'):
    res = http_get_json(f'{INAT_BASE}/taxa', {'q': name, 'rank': rank}).get('results') or []
    return res[0]['id'] if res else None

def inat_observations(taxon_id, max_n, leaf_only=True):
    rows, page, per = [], 1, 100
    while len(rows) < max_n:
        params = {'taxon_id': taxon_id, 'quality_grade': 'research', 'photos': 'true',
                  'per_page': per, 'page': page, 'order': 'desc', 'order_by': 'created_at'}
        if leaf_only:
            params['term_id'] = INAT_LEAF_TERM_ID
            params['term_value_id'] = INAT_LEAF_TERM_VALUE_ID
        batch = (http_get_json(f'{INAT_BASE}/observations', params).get('results') or [])
        if not batch: break
        rows.extend(batch)
        if len(batch) < per: break
        page += 1; time.sleep(0.25)
    return rows[:max_n]

def _photo_url(photo, size='medium'):
    raw = photo.get('url') or ''
    return re.sub(r'/square\.', f'/{size}.', raw) if raw.startswith('http') else None

def _save_one_inat(item, dest, klass, tx, label_confidence):
    obs, ph = item
    url = _photo_url(ph, 'medium')
    if not url: return 0
    fp = dest / f'inat_{tx}_{obs["id"]}_{ph["id"]}.jpg'
    if fp.exists(): return 0
    body = http_get_bytes(url)
    img = _valid_image_bytes(body) if body else None
    if img is None: return 0
    _save_rgb(img, fp)
    write_sidecar(fp, CropAnnotation(
        plant_id=str(obs.get('id')),
        species=(obs.get('taxon') or {}).get('name'),
        diagnosis=klass, severity='unknown', leaf_or_whole_plant='leaf',
        capture_session=f'inat_taxon_{tx}', source='inat',
        label_confidence=label_confidence, bbox=[0, 0, 0, 0]))
    return 1

def download_inat_class(taxon_ids, dest, klass, max_per_taxon=160, leaf_only=True,
                        workers=8, label_confidence=0.7):
    dest.mkdir(parents=True, exist_ok=True)
    n = 0
    for tx in taxon_ids:
        try:
            obs_list = inat_observations(tx, max_per_taxon, leaf_only=leaf_only)
        except Exception as e:
            print(f'  iNat taxon {tx} failed: {e}'); continue
        items = [(o, p) for o in obs_list for p in (o.get('photos') or [])[:1]]
        if not items: continue
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futs = [pool.submit(_save_one_inat, it, dest, klass, tx, label_confidence) for it in items]
            for f in as_completed(futs):
                n += f.result()
    return n

INAT_HEALTHY_TAXA = [
    ('Ficus','genus'),('Monstera','genus'),('Epipremnum','genus'),('Calathea','genus'),
    ('Spathiphyllum','genus'),('Dracaena','genus'),('Sansevieria','genus'),('Philodendron','genus'),
]
INAT_BOTRYTIS_TAXA = [('Botrytis cinerea','species'),('Botrytis','genus')]

def fetch_inat_supplement(healthy_per_taxon=180, botrytis_per_taxon=160):
    healthy_ids = [t for t in (inat_resolve_taxon(n, r) for n, r in INAT_HEALTHY_TAXA) if t]
    n = download_inat_class(healthy_ids, TRAIN_ROOT / 'healthy', 'healthy',
                            max_per_taxon=healthy_per_taxon, leaf_only=True,
                            label_confidence=0.75)
    print(f'iNat healthy += {n}')
    # Botrytis is a fungus, not a plant. iNat's leaf-only filter
    # (term_id=36 'Plant Phenology' value=38 'Green Leaves') is
    # populated for plant observations, not fungus ones — applying it
    # here returns ~0 hits and the THRESH_TRAIN guard fires. Fungal
    # observations on iNat are already photos OF the fungus on a host,
    # which is exactly what we want.
    botrytis_ids = [t for t in (inat_resolve_taxon(n, r) for n, r in INAT_BOTRYTIS_TAXA) if t]
    n = download_inat_class(botrytis_ids, TRAIN_ROOT / 'gray_mold_botrytis', 'gray_mold_botrytis',
                            max_per_taxon=botrytis_per_taxon, leaf_only=False,
                            label_confidence=0.55)
    print(f'iNat gray_mold_botrytis supplement += {n}')

# ── COCO val2017 → not_a_plant ──────────────────────────────────
COCO_CACHE = ROOT / 'cache_coco_val2017'
def fetch_coco_for_nap(target_n=1500):
    COCO_CACHE.mkdir(parents=True, exist_ok=True)
    img_zip, ann_zip = COCO_CACHE/'val2017.zip', COCO_CACHE/'annotations.zip'
    if not (COCO_CACHE/'val2017').exists():
        print('Downloading COCO val2017 (~1 GB)…')
        http_download('http://images.cocodataset.org/zips/val2017.zip', img_zip)
        with zipfile.ZipFile(img_zip) as z: z.extractall(COCO_CACHE)
    if not (COCO_CACHE/'annotations').exists():
        http_download('http://images.cocodataset.org/annotations/annotations_trainval2017.zip', ann_zip)
        with zipfile.ZipFile(ann_zip) as z: z.extractall(COCO_CACHE)
    ann = json.loads((COCO_CACHE/'annotations'/'instances_val2017.json').read_text())
    plant_like_names = {'potted plant','vase','broccoli','carrot','banana','apple','orange'}
    plant_cat = {c['id'] for c in ann['categories'] if c['name'] in plant_like_names}
    plant_imgs = {a['image_id'] for a in ann['annotations'] if a['category_id'] in plant_cat}
    eligible = [im for im in ann['images'] if im['id'] not in plant_imgs]
    rng = random.Random(42); rng.shuffle(eligible)
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    n = 0
    for im in eligible[:target_n]:
        src, dst = COCO_CACHE/'val2017'/im['file_name'], dest/f'coco_{im["id"]}.jpg'
        if not src.exists() or dst.exists(): continue
        shutil.copy(src, dst)
        write_sidecar(dst, CropAnnotation(plant_id=f'coco_{im["id"]}', species=None,
            diagnosis='not_a_plant', severity='low', leaf_or_whole_plant='whole_plant',
            capture_session='coco_val2017', source='coco', label_confidence=0.99, bbox=[0,0,0,0]))
        n += 1
    print(f'COCO: not_a_plant += {n}')

# ── PlantDoc (HF) → real-world train + val_realworld seed ───────
def map_plantdoc_label(raw):
    s = re.sub(r'\s+', ' ', str(raw).strip().lower())
    if 'spider mite' in s or 'mites' in s:
        return None
    if 'bacterial spot' in s or 'bacterial blight' in s:
        return 'bacterial_leaf_spot'
    if 'powdery mildew' in s:
        return 'powdery_mildew'
    if 'yellow virus' in s or 'mosaic virus' in s or 'curl virus' in s:
        return 'viral_disease'
    if any(k in s for k in ['blight','rust','scab','leaf spot','mold','rot','anthracnose']):
        return 'leaf_spot_fungal'
    if 'leaf' in s:
        return 'healthy'
    return None

def _clip_bbox_xywh(bbox, w, h, pad=0.06):
    """Clip an XYWH (COCO-style) bbox to image bounds with padding.

    Sanity-checks the format before trusting it: a [x1, y1, x2, y2]
    XYXY tuple sneaking through the loader would otherwise look like
    bw=x2 / bh=y2 — clip would clamp to most of the image and silently
    train on whole-frame crops instead of leaf crops.
    """
    x, y, bw, bh = [float(v) for v in bbox]
    if bw <= 1 or bh <= 1 or w <= 1 or h <= 1:
        return None
    # XYXY heuristic: in XYWH, (x + bw) and (y + bh) must stay inside
    # the image (with a small slack for off-by-one annotations). If both
    # `bw > w` AND `bh > h`, the annotator likely meant XYXY, not XYWH.
    if bw > w * 1.05 and bh > h * 1.05:
        return None
    px, py = bw * pad, bh * pad
    x0, y0 = max(0, int(round(x - px))), max(0, int(round(y - py)))
    x1, y1 = min(w, int(round(x + bw + px))), min(h, int(round(y + bh + py)))
    if x1 - x0 < 32 or y1 - y0 < 32:
        return None
    return x0, y0, x1, y1

def _plantdoc_image_label(row, ds_split):
    """Resolve a PlantDoc image-level class label as a string.

    Used as a fallback when the HF revision ships image-level labels
    instead of bbox-level ``objects``. Mirrors the heuristics in
    ``_row_label`` for PlantVillage.
    """
    if 'label' in row:
        feat = ds_split.features.get('label') if hasattr(ds_split, 'features') else None
        if hasattr(feat, 'names') and feat.names:
            return feat.names[int(row['label'])]
        return row.get('label_name') or str(row['label'])
    return row.get('label_name') or row.get('class') or row.get('disease')

def fetch_plantdoc(max_train_per_class=500, max_val_per_class=120):
    try:
        from datasets import load_dataset
        ds = load_dataset('agyaatcoder/PlantDoc', token=False)
    except Exception as e:
        print(f'PlantDoc HF download failed: {e}'); return
    counts, dropped, no_obj_fallback = {}, 0, 0
    for split in [s for s in ('train', 'test') if s in ds]:
        out_base = VAL_PRIMARY if split == 'test' else TRAIN_ROOT
        cap = max_val_per_class if split == 'test' else max_train_per_class
        for i, row in enumerate(tqdm(ds[split], desc=f'PlantDoc/{split}')):
            img = row['image'].convert('RGB')
            w, h = img.size
            objects = row.get('objects') or {}
            bboxes = objects.get('bbox') or []
            cats = objects.get('category') or []
            # Object-level path (preferred — HF revisions ≥ 1.x).
            wrote_any = False
            for j, (bbox, cat) in enumerate(zip(bboxes, cats)):
                target = map_plantdoc_label(cat)
                if target is None or target not in CLASSES:
                    dropped += 1; continue
                key = (split, target)
                if counts.get(key, 0) >= cap:
                    continue
                box = _clip_bbox_xywh(bbox, w, h)
                if box is None: dropped += 1; continue
                crop = img.crop(box)
                image_id = row.get('image_id', i)
                stem = f'plantdoc_{split}_{image_id}_{j}'
                out = out_base / target / f'{stem}.jpg'
                if out.exists(): wrote_any = True; continue
                _save_rgb(crop, out)
                write_sidecar(out, CropAnnotation(
                    plant_id=f'plantdoc_{split}_{image_id}',
                    species=str(cat).split(' leaf')[0],
                    diagnosis=target, severity='unknown', leaf_or_whole_plant='leaf',
                    capture_session=f'plantdoc_{split}', source='plantdoc',
                    label_confidence=0.85, bbox=[int(v) for v in bbox]))
                counts[key] = counts.get(key, 0) + 1
                wrote_any = True
            if wrote_any:
                continue
            # Image-level fallback. Triggered when the HF revision drops
            # the object-detection annotations and ships only one
            # category per image (or when every bbox failed the XYWH
            # sanity check). Without this, a schema flip silently halves
            # the training set and the THRESH_TRAIN guard fires three
            # downloads later.
            raw = _plantdoc_image_label(row, ds[split])
            target = map_plantdoc_label(raw) if raw is not None else None
            if target is None or target not in CLASSES:
                dropped += 1; continue
            key = (split, target)
            if counts.get(key, 0) >= cap:
                continue
            image_id = row.get('image_id', i)
            stem = f'plantdoc_{split}_{image_id}_full'
            out = out_base / target / f'{stem}.jpg'
            if out.exists(): continue
            _save_rgb(img, out)
            write_sidecar(out, CropAnnotation(
                plant_id=f'plantdoc_{split}_{image_id}',
                species=str(raw).split(' leaf')[0] if raw else None,
                diagnosis=target, severity='unknown', leaf_or_whole_plant='whole_plant',
                capture_session=f'plantdoc_{split}', source='plantdoc',
                label_confidence=0.75, bbox=[0, 0, 0, 0]))
            counts[key] = counts.get(key, 0) + 1
            no_obj_fallback += 1
    print(f'PlantDoc counts: {counts}\nPlantDoc dropped: {dropped}'
          f'  image-level fallbacks: {no_obj_fallback}')

# ── PlantVillage (HF) → capped train + val_studio ───────────────
def _pv_normalise(s):
    return re.sub(r'[^a-z0-9_]+', '_', str(s).strip().lower()).strip('_')

PV_LABEL_MAP_RAW = {
    'Pepper,_bell___Bacterial_spot': 'bacterial_leaf_spot',
    'Pepper__bell___Bacterial_spot': 'bacterial_leaf_spot',
    'Tomato___Bacterial_spot':       'bacterial_leaf_spot',
    'Peach___Bacterial_spot':        'bacterial_leaf_spot',
    'Apple___healthy': 'healthy', 'Blueberry___healthy': 'healthy',
    'Cherry_(including_sour)___healthy': 'healthy', 'Corn_(maize)___healthy': 'healthy',
    'Grape___healthy': 'healthy',
    'Pepper,_bell___healthy': 'healthy', 'Pepper__bell___healthy': 'healthy',
    'Potato___healthy': 'healthy', 'Raspberry___healthy': 'healthy',
    'Soybean___healthy': 'healthy', 'Strawberry___healthy': 'healthy',
    'Tomato___healthy': 'healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'leaf_spot_fungal',
    'Tomato___Septoria_leaf_spot': 'leaf_spot_fungal',
    'Tomato___Target_Spot':        'leaf_spot_fungal',
    'Tomato___Leaf_Mold':          'leaf_spot_fungal',
    'Tomato___Early_blight':       'leaf_spot_fungal',
    'Tomato___Late_blight':        'leaf_spot_fungal',
    'Potato___Early_blight':       'leaf_spot_fungal',
    'Potato___Late_blight':        'leaf_spot_fungal',
    'Apple___Apple_scab':          'leaf_spot_fungal',
    'Apple___Black_rot':           'leaf_spot_fungal',
    'Apple___Cedar_apple_rust':    'leaf_spot_fungal',
    'Corn_(maize)___Northern_Leaf_Blight': 'leaf_spot_fungal',
    'Corn_(maize)___Common_rust_': 'leaf_spot_fungal',
    'Grape___Black_rot':           'leaf_spot_fungal',
    'Grape___Esca_(Black_Measles)':'leaf_spot_fungal',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 'leaf_spot_fungal',
    'Orange___Haunglongbing_(Citrus_greening)':   None,   # HLB ≠ leaf-spot
    'Cherry_(including_sour)___Powdery_mildew': 'powdery_mildew',
    'Squash___Powdery_mildew':                  'powdery_mildew',
    'Tomato___Tomato_mosaic_virus':           'viral_disease',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 'viral_disease',
    'Tomato___Spider_mites Two-spotted_spider_mite': None,
    'Strawberry___Leaf_scorch':  'leaf_spot_fungal',
    'Background_without_leaves': 'not_a_plant',
}
PV_LABEL_MAP = {_pv_normalise(k): v for k, v in PV_LABEL_MAP_RAW.items()}
# Each candidate is (hf_repo, hf_config_name_or_None, splits_tuple). The
# config name is required for mohanty/PlantVillage because that repo ships
# three configs ('color', 'grayscale', 'segmented') and load_dataset() now
# refuses to pick one for you. BrandonFors/... has a single default config,
# so the explicit name is None there.
PV_DATASET_CANDIDATES = [
    ('BrandonFors/Plant-Diseases-PlantVillage-Dataset', None,    ('train', 'test')),
    ('mohanty/PlantVillage',                            'color', ('train', 'test')),
]

def _load_first_available_hf_dataset(candidates):
    from datasets import load_dataset
    errors = []
    for entry in candidates:
        # Backward-compatible with the old 2-tuple form, just in case.
        if len(entry) == 2:
            name, splits = entry; config_name = None
        else:
            name, config_name, splits = entry
        try:
            out = {}
            for split in [s for s in splits if s]:
                out[split] = load_dataset(name, config_name, split=split, token=False)
            label = name + (f' [{config_name}]' if config_name else '')
            print(f'PlantVillage source: {label}')
            return name, out
        except Exception as e:
            errors.append(f'{name}: {e}')
    raise RuntimeError('All PlantVillage HF downloads failed:\n' + '\n'.join(errors))

def _row_label(row, ds):
    if 'label' in row:
        feat = ds.features.get('label') if hasattr(ds, 'features') else None
        if hasattr(feat, 'names') and feat.names:
            return feat.names[int(row['label'])]
        return row.get('label_name') or str(row['label'])
    return row.get('label_name') or row.get('class') or row.get('disease')

def fetch_plantvillage_into_train_and_studio(max_train_per_class=900, max_studio_per_class=300, studio_hash_frac=0.15):
    try:
        name, dsets = _load_first_available_hf_dataset(PV_DATASET_CANDIDATES)
    except Exception as e:
        print(f'PlantVillage HF download failed: {e}'); return
    counts, dropped = {}, 0
    for split, ds in dsets.items():
        order = list(range(len(ds)))
        random.Random(SEED + (0 if split == 'train' else 1)).shuffle(order)
        for i in tqdm(order, desc=f'PlantVillage/{split}'):
            row = ds[i]
            raw = _row_label(row, ds)
            norm = _pv_normalise(raw)
            target = PV_LABEL_MAP.get(norm)
            if target is None or target not in CLASSES:
                dropped += 1; continue
            filename = row.get('image/filename') or row.get('filename') or f'{split}_{i}'
            stem = f'pv_{_pv_normalise(filename)}_{i}'
            use_studio = (split != 'train') or (_stable_frac(stem) < studio_hash_frac)
            out_base = VAL_STUDIO if use_studio else TRAIN_ROOT
            cap = max_studio_per_class if use_studio else max_train_per_class
            key = ('studio' if use_studio else 'train', target)
            if counts.get(key, 0) >= cap:
                continue
            out = out_base / target / f'{stem}.jpg'
            if out.exists(): continue
            img = row['image']
            if not hasattr(img, 'convert'):
                img = PILImage.open(img)
            _save_rgb(img.convert('RGB'), out)
            write_sidecar(out, CropAnnotation(
                plant_id=stem, species=str(raw).split('___')[0].replace('_', ' '),
                diagnosis=target, severity='unknown', leaf_or_whole_plant='leaf',
                capture_session=f'plantvillage_{split}', source='plantvillage',
                label_confidence=0.9, bbox=[0, 0, 0, 0]))
            counts[key] = counts.get(key, 0) + 1
    print(f'PV counts: {counts}\nPV dropped/ignored: {dropped}')

# ── Group-aware splits ──────────────────────────────────────────
def make_group_split(dest_root, ratio, seed, label, allowed_sources=None):
    """Move a fraction of TRAIN_ROOT into dest_root.

    Same `(source, plant_id)` lands together. `allowed_sources`
    constrains which sources can populate the split — used to keep
    PlantVillage out of the real-world / calibration splits.
    """
    rng = random.Random(seed); moved = 0
    for cls in CLASSES:
        cls_dir = TRAIN_ROOT / cls
        if not cls_dir.exists(): continue
        groups = {}
        for img in iter_images(cls_dir):
            sc = read_sidecar(img) or {}
            source = sc.get('source', 'unknown')
            if allowed_sources is not None and source not in allowed_sources:
                continue
            pid = sc.get('plant_id', img.stem)
            groups.setdefault((source, pid), []).append(img)
        ids = sorted(groups); rng.shuffle(ids)
        k = int(len(ids) * ratio)
        if ratio > 0 and len(ids) > 0:
            k = max(1, k)
        out = dest_root / cls; out.mkdir(parents=True, exist_ok=True)
        for gid in ids[:k]:
            for img in groups[gid]:
                dst = out / img.name
                if dst.exists():
                    raise RuntimeError(f'Destination already exists: {dst}')
                shutil.move(str(img), str(dst))
                sc_path = img.with_suffix('.json')
                if sc_path.exists():
                    shutil.move(str(sc_path), str(dst.with_suffix('.json')))
                moved += 1
    print(f'{label} populated  (moved={moved})')

def make_val_realworld(ratio=0.15, seed=2):
    make_group_split(VAL_PRIMARY, ratio, seed, 'val_realworld',
                     allowed_sources={'inat', 'coco', 'plantdoc'})
def make_val_indomain(ratio=0.10, seed=0):
    # Mirror the realworld / cal split rule: keep PV studio shots out
    # of val_indomain so early-stopping doesn't pick PV-favoring epochs.
    make_group_split(VAL_INDOMAIN, ratio, seed, 'val_indomain',
                     allowed_sources={'inat', 'coco', 'plantdoc'})
def make_temperature_cal(ratio=0.05, seed=3):
    make_group_split(TEMP_CAL, ratio, seed, 'temperature_cal',
                     allowed_sources={'plantdoc', 'inat'})
def make_conformal_cal(ratio=0.10, seed=1):
    make_group_split(CONFORMAL_CAL, ratio, seed, 'conformal_cal',
                     allowed_sources={'plantdoc', 'inat'})

# ── Run the pipeline ────────────────────────────────────────────
fetch_inat_supplement(healthy_per_taxon=180, botrytis_per_taxon=160)
fetch_coco_for_nap(target_n=1500)
fetch_plantdoc(max_train_per_class=500, max_val_per_class=120)
fetch_plantvillage_into_train_and_studio(max_train_per_class=900, max_studio_per_class=300)
make_val_realworld(); make_val_indomain(); make_temperature_cal(); make_conformal_cal()

def report_class_counts(root):
    print(f'\n{root}:')
    for c in CLASSES:
        print(f'  {c:25s} {count_images(root/c):5d}')
for r in (TRAIN_ROOT, VAL_PRIMARY, VAL_INDOMAIN, TEMP_CAL, CONFORMAL_CAL, VAL_STUDIO):
    report_class_counts(r)

# Hard guard. Empty class folders silently produce recall=0 and skew
# priors. Every shipped class must have train + primary-gate support.
THRESH_TRAIN, THRESH_VAL = 30, 5
under_train = [c for c in CLASSES if count_images(TRAIN_ROOT/c) < THRESH_TRAIN]
under_val   = [c for c in CLASSES if count_images(VAL_PRIMARY/c) < THRESH_VAL]
if under_train or under_val:
    msg = []
    if under_train: msg.append(f'TRAIN_ROOT < {THRESH_TRAIN}: {under_train}')
    if under_val:   msg.append(f'VAL_PRIMARY < {THRESH_VAL}: {under_val}')
    raise RuntimeError('\n'.join(msg) +
        '\nAdd / extend a public source for these classes and rerun §2.')


## §3 — Dedup + Stage 1 detector + DINOv2 diagnostic probe

Three sanity passes before we burn GPU on the teacher:

1. **Dedup** the train set (per-class SHA-256 + pHash with Hamming ≤ 6).
   Cross-class similarities are kept on purpose — they are useful as
   hard negatives. Held-out splits are audited (SHA-256 only) but
   never pruned, because pruning a held-out set after the fact biases
   the gate.

2. **Stage 1 YOLOv8n** is re-trained only when a YOLO dataset is
   present under `data_v2/` (driven by `find_detector_yaml()`). When
   nothing is found we keep the shipped `detector.{pt,onnx}` pair —
   the bigger time win is data hygiene, not yet-another detector run.

3. **DINOv2-S linear probe** is the cheap "is the data even
   separable?" check. If linear-on-frozen-DINOv2 cannot get past 30 %
   on `val_realworld` after 8 epochs, the data is the problem, not
   the model.


In [ ]:
# §3 — Dedup + Stage 1 detector + DINOv2 probe
import imagehash
from PIL import Image as PILImage
from ultralytics import YOLO
import yaml

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

def dedup_directory(root, phash_threshold=6):
    """Per-class SHA-256 + pHash dedup."""
    drop = []
    for cls in CLASSES:
        cls_dir = Path(root) / cls
        if not cls_dir.exists(): continue
        seen_sha, seen_phash = set(), []
        for p in sorted(iter_images(cls_dir)):
            s = sha256_file(p)
            if s in seen_sha:
                drop.append((p, 'sha256_dup')); continue
            seen_sha.add(s)
            try:
                ph = imagehash.phash(PILImage.open(p).convert('RGB'))
            except Exception:
                drop.append((p, 'unreadable')); continue
            if any(ph - existing <= phash_threshold for existing in seen_phash):
                drop.append((p, 'phash_dup')); continue
            seen_phash.append(ph)
    for p, _reason in drop:
        sc = p.with_suffix('.json')
        try:
            p.unlink()
            if sc.exists(): sc.unlink()
        except Exception:
            pass
    print(f'dedup({root}): dropped={len(drop)}')

def assert_no_exact_split_leakage(train_root, heldout_roots):
    """SHA-256 cross-check between train and every held-out split."""
    train_hash = {sha256_file(p): p for p in iter_images(train_root)}
    problems = []
    for root in heldout_roots:
        if not Path(root).exists(): continue
        for p in iter_images(root):
            h = sha256_file(p)
            if h in train_hash:
                problems.append((str(train_hash[h]), str(p)))
                if len(problems) >= 10: break
        if len(problems) >= 10: break
    if problems:
        detail = '\n'.join(f'train={a}  heldout={b}' for a, b in problems)
        raise RuntimeError('Exact train/heldout leakage detected:\n' + detail)
    print('leakage audit: no exact SHA overlap between train and held-out splits')

dedup_directory(TRAIN_ROOT)
assert_no_exact_split_leakage(TRAIN_ROOT, SPLIT_ROOTS)

# ── Stage 1 — YOLOv8n single-class plant_leaf ───────────────────
def find_detector_yaml():
    if not ROOT.exists(): return None
    cands = [p for p in ROOT.rglob('data.yaml') if 'cache_coco' not in str(p)]
    return cands[0] if cands else None

def _yolo_label_ids(label_dir):
    ids = set()
    for txt in Path(label_dir).rglob('*.txt'):
        for line in txt.read_text().splitlines():
            parts = line.strip().split()
            if parts:
                try: ids.add(int(float(parts[0])))
                except ValueError: pass
    return ids

yaml_path = find_detector_yaml()
if yaml_path is None:
    print(f'No YOLO dataset under {ROOT}.  Stage 1 training skipped — '
          f'the shipped detector.{{pt,onnx}} will be used at inference.')
else:
    spec = yaml.safe_load(yaml_path.read_text())
    for key in ('train', 'val'):
        p = spec.get(key)
        if isinstance(p, str):
            labels_dir = Path(yaml_path.parent, p).as_posix().replace('/images/', '/labels/')
            if Path(labels_dir).exists():
                ids = _yolo_label_ids(labels_dir)
                if any(i != 0 for i in ids):
                    raise RuntimeError(
                        f'{key} labels contain class ids {sorted(ids)}; remap to single class 0 first.'
                    )
    spec['names'], spec['nc'] = ['plant_leaf'], 1
    patched = yaml_path.parent / 'plant_leaf.yaml'
    patched.write_text(yaml.safe_dump(spec, sort_keys=False))
    detector = YOLO('yolov8n.pt')
    detector.train(
        data=str(patched), epochs=60, patience=12, batch=32, imgsz=640,
        optimizer='AdamW', lr0=1e-3, cos_lr=True, amp=True,
        # Indoor leaf bboxes are tight — heavy mosaic = leaks across boxes.
        mosaic=0.1, hsv_h=0.0, hsv_s=0.4, hsv_v=0.4,
        project='runs/detect', name='plant_leaf_indoor_v1', exist_ok=True,
    )
    best_pt = Path(detector.trainer.best) if getattr(detector, 'trainer', None) else next(
        Path('runs/detect').rglob('best.pt'))
    exported = YOLO(str(best_pt)).export(format='onnx', opset=13, dynamic=False)
    Path('models').mkdir(exist_ok=True)
    shutil.copy(best_pt, 'models/detector.pt')
    if exported and Path(exported).exists():
        shutil.copy(exported, 'models/detector.onnx')
    else:
        src = next(best_pt.parent.rglob('*.onnx'), None)
        if src: shutil.copy(src, 'models/detector.onnx')

# ── DINOv2-S linear probe (diagnostic) ──────────────────────────
PROBE_SIZE = 224
PROBE_TX = T.Compose([T.Resize(int(PROBE_SIZE * 1.05)),
                      T.CenterCrop(PROBE_SIZE), T.ToTensor(), NORM])
train_ds_probe = ImageFolder(str(TRAIN_ROOT), transform=PROBE_TX)
assert train_ds_probe.classes == sorted(CLASSES), (train_ds_probe.classes, sorted(CLASSES))
TRAIN_CLASS_TO_IDX = dict(train_ds_probe.class_to_idx)
train_dl_probe = DataLoader(
    train_ds_probe, batch_size=256, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    **_loader_kwargs(EVAL_WORKERS, drop_last=False),
)
_, val_dl_probe = make_eval_loader(
    VAL_PRIMARY, PROBE_SIZE, batch=256, class_to_idx=TRAIN_CLASS_TO_IDX,
)

backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').eval().to(device)
for p in backbone.parameters(): p.requires_grad_(False)
head = nn.Linear(384, len(train_ds_probe.classes)).to(device)
opt = torch.optim.AdamW(head.parameters(), lr=3e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

probe_curve = []
for ep in range(8):
    head.train()
    for x, y in train_dl_probe:
        x, y = x.to(device, non_blocking=True), y.to(device)
        with torch.no_grad():
            f = backbone(x)
        opt.zero_grad(set_to_none=True)
        loss_fn(head(f), y).backward()
        opt.step()
    head.eval()
    correct = total = 0
    with torch.inference_mode():
        for x, y in val_dl_probe:
            correct += (head(backbone(x.to(device))).argmax(1) == y.to(device)).sum().item()
            total += y.numel()
    top1 = correct / max(total, 1)
    probe_curve.append({'epoch': ep + 1, 'val_realworld_top1': top1})
    print(f'ep={ep+1}  val_realworld_top1={top1:.3f}')

Path('models/dinov2_probe.json').write_text(json.dumps(probe_curve, indent=2))
log_run({'phase': 'dinov2_probe', 'curve': probe_curve})
if probe_curve[-1]['val_realworld_top1'] < 0.30:
    print('WARN: DINOv2 frozen+probe < 0.30 — your data is hard. '
          'Audit labels in §2 before spending GPU on the teacher.')

# Free memory before §4.
del backbone, head, train_dl_probe, val_dl_probe
torch.cuda.empty_cache() if device == 'cuda' else None


## §4 — Loss, augs, teacher fine-tune (EfficientNetV2-S)

* **Loss:** `LogitAdjustmentLoss(τ=0.7, label_smoothing=0.05)`
  (Menon et al. ICLR 2021). Loss-side rebalancing matched to the
  **training prior** — no `WeightedRandomSampler` by default, because
  stacking the two over-corrects rare classes (Liu et al. 2024,
  *frontiersin*).
* **Sampler off by default.** Set `USE_WEIGHTED_SAMPLER = True` only
  if a single class dominates by > 10×.
* **Two-phase fine-tune:** head-only for 3 epochs (LR=3e-3), then
  unfreeze the last two backbone stages (head LR=5e-4, backbone
  LR=5e-5), early-stop on `val_indomain` disease macro-F1.
* **Augs:** indoor-targeted — WB shift, glare, defocus / motion blur,
  JPEG re-compression, mild ColorJitter, RandomErasing. **No** strong
  hue rotation / Solarize — colour is a disease signal.


In [ ]:
# §4 — Loss, augs, teacher fine-tune
import timm, torch.nn.functional as F
from collections import Counter
from contextlib import nullcontext
from torch.amp import autocast
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import WeightedRandomSampler
from PIL import ImageFilter, ImageEnhance, Image as PILImage
from io import BytesIO

USE_WEIGHTED_SAMPLER = False   # off by default — see notebook §4 docstring
LA_TAU = 0.7                   # logit adjustment τ
LABEL_SMOOTHING = 0.05

class LogitAdjustmentLoss(nn.Module):
    """Loss-side Logit Adjustment, Menon et al. ICLR 2021 eq. 4."""
    def __init__(self, class_priors, tau=LA_TAU, label_smoothing=LABEL_SMOOTHING):
        super().__init__()
        self.register_buffer('log_prior', torch.log(class_priors.clamp(min=1e-12)))
        self.tau, self.ls = tau, label_smoothing
    def forward(self, logits, target):
        return F.cross_entropy(logits + self.tau * self.log_prior, target,
                               label_smoothing=self.ls)

def class_priors_from_imagefolder(ds):
    counts = Counter([y for _, y in ds.samples])
    pri = torch.tensor([counts.get(i, 0) for i in range(len(ds.classes))], dtype=torch.float32)
    return pri.clamp(min=1.0) / pri.clamp(min=1.0).sum()

def make_class_balanced_sampler(ds, power=0.5):
    counts = Counter([y for _, y in ds.samples])
    weights = torch.DoubleTensor([(1.0 / counts[y]) ** power for _, y in ds.samples])
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True,
                                 generator=torch.Generator().manual_seed(SEED))

def amp_ctx():
    return autocast(device_type='cuda', dtype=torch.bfloat16) if device == 'cuda' else nullcontext()

class WhiteBalanceShift:
    def __init__(self, p=0.5, strength=(0.85, 1.15)):
        self.p, self.lo, self.hi = p, *strength
    def __call__(self, img):
        if random.random() > self.p: return img
        r_scale = random.uniform(self.lo, self.hi)
        b_scale = random.uniform(self.lo, self.hi)
        r, g, b = img.split()
        r = r.point(lambda v: max(0, min(255, int(v * r_scale))))
        b = b.point(lambda v: max(0, min(255, int(v * b_scale))))
        return PILImage.merge('RGB', (r, g, b))

class Glare:
    def __init__(self, p=0.25): self.p = p
    def __call__(self, img):
        return img if random.random() > self.p else \
               ImageEnhance.Brightness(img).enhance(random.uniform(1.05, 1.35))

class DefocusOrMotion:
    def __init__(self, p=0.3): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        f = (ImageFilter.GaussianBlur(radius=random.uniform(0.6, 1.8))
             if random.random() < 0.5
             else ImageFilter.BoxBlur(random.uniform(0.8, 2.0)))
        return img.filter(f)

class JpegRecompress:
    def __init__(self, p=0.35, q_range=(50, 92)): self.p, self.q_range = p, q_range
    def __call__(self, img):
        if random.random() > self.p: return img
        buf = BytesIO()
        img.save(buf, format='JPEG', quality=random.randint(*self.q_range))
        buf.seek(0)
        return PILImage.open(buf).convert('RGB').copy()

indoor_train_tx = T.Compose([
    T.Resize(int(256 * 1.15)),
    T.RandomResizedCrop(256, scale=(0.70, 1.0), ratio=(0.88, 1.14)),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.RandomPerspective(distortion_scale=0.12, p=1.0)], p=0.20),
    WhiteBalanceShift(p=0.5), Glare(p=0.25), DefocusOrMotion(p=0.3), JpegRecompress(p=0.35),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.0),
    T.ToTensor(), NORM, T.RandomErasing(p=0.12, scale=(0.02, 0.08)),
])

# ── Teacher: EfficientNetV2-S ──────────────────────────────────
BATCH, IMG = 96, 256       # 96 fits comfortably on Colab T4 / A100
train_ds = ImageFolder(str(TRAIN_ROOT), transform=indoor_train_tx)
if len(train_ds) == 0:
    raise RuntimeError('TRAIN_ROOT empty — re-run §2.')
CLASS_NAMES = list(train_ds.classes)
assert CLASS_NAMES == sorted(CLASSES), (CLASS_NAMES, sorted(CLASSES))
TRAIN_CLASS_TO_IDX = dict(train_ds.class_to_idx)
N_CLASSES = len(CLASS_NAMES)

priors = class_priors_from_imagefolder(train_ds).to(device)
loss_fn = LogitAdjustmentLoss(priors, tau=LA_TAU, label_smoothing=LABEL_SMOOTHING)

if USE_WEIGHTED_SAMPLER:
    sampler = make_class_balanced_sampler(train_ds, power=0.5)
    train_dl = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                          **_loader_kwargs(min(8, EVAL_WORKERS), drop_last=True))
else:
    train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED),
                          **_loader_kwargs(min(8, EVAL_WORKERS), drop_last=True))

val_indomain_ds, val_indomain_dl = make_eval_loader(
    VAL_INDOMAIN, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
val_realworld_ds, val_realworld_dl = make_eval_loader(
    VAL_PRIMARY, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
temp_cal_ds, temp_cal_dl = make_eval_loader(
    TEMP_CAL, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
try:
    val_studio_ds, val_studio_dl = make_eval_loader(
        VAL_STUDIO, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
    )
except (RuntimeError, FileNotFoundError):
    # torchvision>=0.15 raises FileNotFoundError when no class subdirs
    # exist (e.g. PlantVillage download was skipped). Studio is a
    # transfer monitor — never blocking, so skip cleanly.
    print('val_studio not available (no class subdirs); skipping studio monitor.')
    val_studio_ds = val_studio_dl = None

teacher = timm.create_model(
    'tf_efficientnetv2_s.in21k_ft_in1k', pretrained=True, num_classes=N_CLASSES,
).to(device)

def step_eval(model, tag):
    """Per-epoch eval used for early-stopping decisions only.

    val_realworld is intentionally NOT computed per epoch: it is the
    ship gate, and looking at it every epoch leaks via human inspection
    of run_log.jsonl ('which epoch had the best realworld_f1?'). The
    honest, single, post-training measurement happens in §6 against
    the deployed INT8 ORT session.
    """
    out = {'indomain': evaluate(model, val_indomain_dl, CLASS_NAMES, device, f'{tag} val_indomain')}
    if val_studio_dl is not None:
        out['studio'] = evaluate(model, val_studio_dl, CLASS_NAMES, device, f'{tag} val_studio')
    log_run({'phase': 'teacher', 'tag': tag,
             'indomain_disease_f1': out['indomain']['disease_macro_f1'],
             'indomain_healthy_bias': out['indomain']['healthy_bias']})
    return out

def _make_sched(opt, n_epochs, n_steps_per_epoch):
    if n_steps_per_epoch <= 0:
        raise RuntimeError('Training loader has zero batches; lower BATCH or add data.')
    return SequentialLR(opt, schedulers=[
        LinearLR(opt, start_factor=0.1, total_iters=n_steps_per_epoch),
        CosineAnnealingLR(opt, T_max=max(1, (n_epochs - 1) * n_steps_per_epoch)),
    ], milestones=[n_steps_per_epoch])

def _train_phase(model, opt, n_epochs, scheduler, tag, early_stop=False, patience=4):
    best_f1, best_state, plateau = -1.0, None, 0
    for ep in range(n_epochs):
        model.train()
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device)
            with amp_ctx():
                logits = model(x)
                loss = loss_fn(logits, y)
            if not torch.isfinite(loss):
                print(f'WARN: non-finite loss at {tag} ep{ep+1}; skipping batch'); continue
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], max_norm=5.0)
            opt.step(); scheduler.step()
        m = step_eval(model, f'{tag} ep{ep+1}')
        if early_stop:
            f1 = m['indomain']['disease_macro_f1']
            if f1 > best_f1:
                best_f1, plateau = f1, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                plateau += 1
                if plateau >= patience:
                    print(f'early stop @ ep{ep+1}  (best f1={best_f1:.3f})')
                    break
    if early_stop and best_state is not None:
        model.load_state_dict(best_state)
        print(f'restored best teacher weights  best_indomain_f1={best_f1:.3f}')

# Phase 1 — head-only
for n, p in teacher.named_parameters():
    p.requires_grad_('classifier' in n)
opt = torch.optim.AdamW([p for p in teacher.parameters() if p.requires_grad],
                        lr=3e-3, weight_decay=1e-4)
_train_phase(teacher, opt, 3, _make_sched(opt, 3, len(train_dl)), 'head')

# Phase 2 — unfreeze last two backbone stages
block_idxs = sorted({int(m.group(1)) for n in dict(teacher.named_parameters())
                     if (m := re.match(r'blocks\.(\d+)\.', n))})
if not block_idxs:
    raise RuntimeError('teacher has no `blocks.N.*` parameters — backbone topology changed.')
top_blocks = [f'blocks.{i}' for i in block_idxs[-2:]]
unfreeze = top_blocks + ['classifier']
for p in teacher.parameters(): p.requires_grad_(False)
for n, p in teacher.named_parameters():
    if any(k in n for k in unfreeze): p.requires_grad_(True)
head_params = [p for n, p in teacher.named_parameters() if 'classifier' in n and p.requires_grad]
backbone_params = [p for n, p in teacher.named_parameters()
                   if any(k in n for k in top_blocks) and p.requires_grad]
if not head_params or not backbone_params:
    raise RuntimeError(f'empty optimiser group: head={len(head_params)} backbone={len(backbone_params)}')
print(f'Phase 2 unfreeze: {top_blocks} + classifier')
opt = torch.optim.AdamW([
    {'params': head_params,     'lr': 5e-4},
    {'params': backbone_params, 'lr': 5e-5},
], weight_decay=1e-4)
_train_phase(teacher, opt, 15, _make_sched(opt, 15, len(train_dl)), 'tune', early_stop=True)


## §5 — Knowledge distillation into MobileNetV3-Large

* **Loss:** `α · KD + (1-α) · CE_LA`, with α=0.5, τ=4.0. CE keeps the
  hard target signal; KD softens the teacher's confidence into a
  matching distribution. We use the same `LogitAdjustmentLoss` for the
  CE branch so the student inherits the teacher's prior correction.
* **EMA student** (decay=0.999) — small-data regime; the EMA copy is
  what we evaluate, save and export. Vanilla weights are noisy step-
  to-step; EMA smooths them out without retraining.
* **Early-stop on `val_indomain` disease macro-F1**, patience=3.
* **Best EMA state restored** before saving.


In [ ]:
# §5 — KD into MobileNetV3-Large with EMA student
from copy import deepcopy
from torchvision import models

T_KD, ALPHA = 4.0, 0.5
EMA_DECAY = 0.999

student = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
student.classifier[3] = nn.Linear(student.classifier[3].in_features, N_CLASSES)
student = student.to(device)

# EMA copy. Same arch, frozen — updated by exponential moving average
# every step. Lives on the same device for cheap updates.
ema_student = deepcopy(student).eval()
for p in ema_student.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def ema_update(ema, online, decay=EMA_DECAY):
    for p_ema, p_on in zip(ema.parameters(), online.parameters()):
        p_ema.mul_(decay).add_(p_on.detach(), alpha=1.0 - decay)
    for b_ema, b_on in zip(ema.buffers(), online.buffers()):
        b_ema.copy_(b_on)

teacher.eval()
opt = torch.optim.AdamW([
    {'params': student.classifier.parameters(), 'lr': 5e-4},
    {'params': [p for n, p in student.named_parameters() if 'classifier' not in n], 'lr': 5e-5},
], weight_decay=1e-4)
KD_EPOCHS = 10
sched = _make_sched(opt, KD_EPOCHS, len(train_dl))

best_f1, best_ema_state, best_ep, plateau, PATIENCE = -1.0, None, 0, 0, 3
for ep in range(KD_EPOCHS):
    student.train()
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device)
        with amp_ctx():
            with torch.no_grad():
                t_logits = teacher(x)
            s_logits = student(x)
            kd = F.kl_div(
                F.log_softmax(s_logits / T_KD, dim=1),
                F.softmax(t_logits / T_KD, dim=1),
                reduction='batchmean',
            ) * (T_KD * T_KD)
            ce = loss_fn(s_logits, y)
            loss = ALPHA * kd + (1 - ALPHA) * ce
        if not torch.isfinite(loss):
            print(f'WARN: non-finite KD loss at ep{ep+1}; skipping batch'); continue
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=5.0)
        opt.step(); sched.step()
        ema_update(ema_student, student)

    metrics = step_eval(ema_student, f'kd-ema ep{ep+1}')
    f1 = metrics['indomain']['disease_macro_f1']
    if f1 > best_f1:
        best_f1, best_ep, plateau = f1, ep + 1, 0
        best_ema_state = {k: v.detach().cpu().clone() for k, v in ema_student.state_dict().items()}
    else:
        plateau += 1
        if plateau >= PATIENCE:
            print(f'KD early stop @ ep{ep+1}  (best ep={best_ep} f1={best_f1:.3f})')
            break

if best_ema_state is not None:
    ema_student.load_state_dict(best_ema_state)
    student.load_state_dict(best_ema_state)   # online ≡ EMA at export time

Path('models').mkdir(exist_ok=True)
torch.save(student.state_dict(), 'models/classifier.pth')
log_run({'phase': 'kd_done', 'best_indomain_f1': best_f1, 'best_ep': best_ep})
print(f'KD complete  best_indomain_f1={best_f1:.3f}  best_ep={best_ep}')

# Free the teacher — ONNX export only needs the student.
del teacher
torch.cuda.empty_cache() if device == 'cuda' else None


## §6 — Production export: ONNX → static INT8 + temperature + conformal

### Export & quantisation order of operations

1. Export FP32 ONNX from the **EMA student weights** (via the online
   model that we synced above). CPU export on purpose — keeps any
   future CUDA-only ops out of the graph.
2. Run `quant_pre_process` (symbolic-shape inference). This is **required**
   — the `try/except` import in v2.0 silently skipped it when the
   helper module was missing, which triggered ORT errors at deploy time.
3. Static INT8 (QDQ) calibration uses **`temperature_cal`**.
   Re-using `conformal_cal` (as v2.1 first attempted) breaks
   exchangeability: q_hat would then be fit on activations that the
   quantiser had already shaped. Per-channel weight quantisation is
   stable on 100–300 representative samples; `temp_cal` is enough.
4. Re-evaluate INT8 ONNX on `val_realworld`. Drop > 0.05 = `RuntimeError`.
   0.02 < drop ≤ 0.05 = warning ship.
5. Fit temperature on **`temperature_cal`** logits collected from the
   INT8 ONNX session (so we calibrate the model we actually deploy).
6. Fit conformal RAPS on **fit-half** of `conformal_cal` (softmax
   probabilities passed through the fitted temperature). Report
   *out-of-sample* coverage on the **audit-half** — in-sample coverage
   is ~1−α by construction and is not a useful number to ship in the
   model card.
7. Final eval on every val slice via the INT8 ORT session.
8. Persist `classes.json`, `temperature.json`, `conformal.json`,
   `model_card.json`, `training_metrics.json`.
9. **Ship gate** runs last — raises `RuntimeError` if any production
   threshold is violated.


In [ ]:
# §6 — ONNX FP32 → static INT8 + temperature + conformal RAPS
import datetime
import onnxruntime as ort
from onnxruntime.quantization import (
    quantize_static, CalibrationDataReader, QuantFormat, QuantType)
try:
    from onnxruntime.quantization.shape_inference import quant_pre_process
except ImportError as e:
    raise RuntimeError(
        'onnxruntime.quantization.shape_inference.quant_pre_process is required. '
        'Upgrade onnxruntime to >=1.17 and re-run §1 install.'
    ) from e

Path('models').mkdir(exist_ok=True)

# 1. ONNX FP32 export from EMA student weights (already loaded into `student`).
student_cpu = student.cpu().eval()
raw_onnx  = 'models/classifier_raw.onnx'
fp32_onnx = 'models/classifier.onnx'
torch.onnx.export(
    student_cpu, torch.randn(1, 3, IMG, IMG), raw_onnx,
    opset_version=18, input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'N'}, 'logits': {0: 'N'}})
quant_pre_process(raw_onnx, fp32_onnx, skip_symbolic_shape=False)
print('ONNX quant_pre_process done')

# 2. Static INT8 calibration from `conformal_cal` (~500 images).
class CalibrationReader(CalibrationDataReader):
    def __init__(self, dataset, n=500, input_name='input'):
        idx = np.random.RandomState(SEED).permutation(len(dataset))[:min(n, len(dataset))]
        self._iter = iter(
            [(dataset[i][0].unsqueeze(0).numpy().astype(np.float32),) for i in idx]
        )
        self.input_name = input_name
    def get_next(self):
        nxt = next(self._iter, None)
        return None if nxt is None else {self.input_name: nxt[0]}

# Calibrate INT8 on TEMP_CAL — the architecture contract reserves
# CONFORMAL_CAL for RAPS only. Re-using the same split for both INT8
# calibration and RAPS would fit q_hat on activations the quantiser
# already shaped, weakening conformal exchangeability. Both splits are
# restricted to the deployment distribution (plantdoc + inat).
quantize_static(
    fp32_onnx, 'models/classifier_int8.onnx',
    calibration_data_reader=CalibrationReader(temp_cal_ds, n=500),
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    per_channel=True,
)
# Conformal split — used only for RAPS quantile fitting in step 5.
cal_dataset, cal_dl = make_eval_loader(
    CONFORMAL_CAL, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)

# 3. Mandatory post-quantization validation on val_realworld.
def _ort_session(path):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(path, sess_options=so, providers=['CPUExecutionProvider'])

def _ort_logits(sess, x_np):
    in_name = sess.get_inputs()[0].name
    return sess.run(None, {in_name: x_np})[0]

def eval_onnx(path, dl):
    sess = _ort_session(path)
    y_true, y_pred = [], []
    for x, y in dl:
        logits = _ort_logits(sess, x.numpy().astype(np.float32))
        y_pred.extend(logits.argmax(1).tolist())
        y_true.extend(y.tolist())
    disease_idx = [i for i, n in enumerate(CLASS_NAMES) if n not in ('healthy', 'not_a_plant')]
    return float(f1_score(y_true, y_pred, labels=disease_idx, average='macro', zero_division=0))

f1_fp32 = eval_onnx(fp32_onnx, val_realworld_dl)
f1_int8 = eval_onnx('models/classifier_int8.onnx', val_realworld_dl)
drift = f1_fp32 - f1_int8
print(f'val_realworld disease F1  fp32={f1_fp32:.4f}  int8={f1_int8:.4f}  drift={drift:+.4f}')
WARN_DRIFT, FAIL_DRIFT = 0.02, 0.05
if drift > FAIL_DRIFT:
    raise RuntimeError(f'INT8 dropped F1 by {drift:.3f} (> {FAIL_DRIFT}). Fall back to FP32.')
elif drift > WARN_DRIFT:
    print(f'WARN: INT8 drift {drift:.3f} > {WARN_DRIFT}; shipping anyway.')

int8_sess = _ort_session('models/classifier_int8.onnx')

def _collect_logits(sess, dl):
    all_logits, all_y = [], []
    for x, y in dl:
        all_logits.append(_ort_logits(sess, x.numpy().astype(np.float32)))
        all_y.append(y.numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)

# 4. Temperature scaling on temperature_cal NLL using INT8 logits.
logits_temp_np, y_temp_np = _collect_logits(int8_sess, temp_cal_dl)
logits_temp = torch.from_numpy(logits_temp_np)
y_temp = torch.from_numpy(y_temp_np).long()

log_T = torch.tensor(0.0, requires_grad=True)
opt_T = torch.optim.LBFGS([log_T], lr=0.05, max_iter=50)
def closure():
    opt_T.zero_grad()
    T_now = log_T.exp().clamp(0.05, 10.0)
    loss = F.cross_entropy(logits_temp / T_now, y_temp)
    loss.backward(); return loss
opt_T.step(closure)
T_val = float(log_T.detach().exp().clamp(0.05, 10.0))
Path('models/temperature.json').write_text(json.dumps({
    'temperature': T_val,
    '_note': 'Fitted with LBFGS on val temperature_cal NLL using INT8 ONNX logits.',
    'fit_scope': 'plant inputs (plantdoc+inat); not_a_plant logits are not represented in temp_cal, so T may be miscalibrated for that class.',
}, indent=2))
print(f'fitted T={T_val:.3f}  '
      f'temp_cal MaxLogit mean={logits_temp.max(1).values.numpy().mean():.2f}')

# 5. Conformal RAPS on conformal_cal probabilities (after temperature).
def _raps_score(p_sorted, lam, k_reg):
    cs = np.cumsum(p_sorted)
    return cs + lam * np.maximum(0.0, np.arange(1, len(p_sorted)+1, dtype=np.float64) - k_reg)

def _eval_raps_set(probs, labels, q_hat, lambda_reg, k_reg):
    n = probs.shape[0]
    if n == 0:
        return float('nan'), float('nan')
    covered, set_sizes = 0, []
    for i in range(n):
        order = np.argsort(-probs[i])
        s = _raps_score(probs[i, order], lambda_reg, k_reg)
        above = np.where(s >= q_hat)[0]
        size = max(int(above[0]) + 1 if above.size else len(s), 1)
        set_sizes.append(size)
        covered += int(labels[i] in order[:size])
    return covered / n, float(np.mean(set_sizes))

def calibrate_q_hat(cal_probs, cal_labels, *, alpha=0.1, lambda_reg=0.01, k_reg=1,
                    audit_probs=None, audit_labels=None):
    n = cal_probs.shape[0]
    if n == 0:
        raise RuntimeError('Empty conformal calibration split.')
    nonconf = np.empty(n)
    for i in range(n):
        order = np.argsort(-cal_probs[i])
        s = _raps_score(cal_probs[i, order], lambda_reg, k_reg)
        nonconf[i] = s[int(np.where(order == cal_labels[i])[0][0])]
    q = float(np.quantile(nonconf, min(1.0, np.ceil((n+1) * (1-alpha)) / n), method='higher'))
    in_cov, in_size = _eval_raps_set(cal_probs, cal_labels, q, lambda_reg, k_reg)
    out_cov = out_size = float('nan'); audit_n = 0
    if audit_probs is not None and audit_probs.shape[0] > 0:
        audit_n = int(audit_probs.shape[0])
        out_cov, out_size = _eval_raps_set(audit_probs, audit_labels, q, lambda_reg, k_reg)
    # `empirical_coverage` keeps its name but reports the honest
    # out-of-sample number when an audit set is supplied; in-sample
    # number stays available alongside for diagnostics.
    primary_cov  = out_cov  if audit_n > 0 else in_cov
    primary_size = out_size if audit_n > 0 else in_size
    return {'method': 'raps', 'alpha': alpha, 'q_hat': q,
            'lambda_reg': lambda_reg, 'k_reg': k_reg,
            'calibration_n': int(n),
            'empirical_coverage': primary_cov,
            'empirical_coverage_kind': 'out_of_sample' if audit_n > 0 else 'in_sample',
            'mean_set_size': primary_size,
            'audit_n': audit_n,
            'in_sample_coverage': in_cov,
            'in_sample_mean_set_size': in_size}

logits_cal_np, y_cal_np = _collect_logits(int8_sess, cal_dl)
probs_cal = torch.softmax(torch.from_numpy(logits_cal_np) / T_val, dim=1).numpy()
# Split conformal_cal 50/50 into a fit-half (q_hat) and an audit-half
# (out-of-sample coverage). Random permutation seeded off SEED so the
# split is deterministic across runs.
n_cal = int(y_cal_np.shape[0])
order_cal = np.random.RandomState(SEED + 7).permutation(n_cal)
fit_n = n_cal - n_cal // 2  # fit-half rounds up
fit_idx, audit_idx = order_cal[:fit_n], order_cal[fit_n:]
cfg = calibrate_q_hat(
    probs_cal[fit_idx], y_cal_np[fit_idx], alpha=0.10,
    audit_probs=probs_cal[audit_idx], audit_labels=y_cal_np[audit_idx],
)
# RAPS exchangeability holds only when calibration ↔ test distributions
# match. conformal_cal is restricted to plantdoc+inat (no COCO / NAP),
# so empirical_coverage applies to plant inputs only. NAP is filtered
# upstream by the Stage 1 detector + MaxLogit OOD gate, never by RAPS.
cfg['coverage_scope'] = 'plant inputs (plantdoc+inat); not_a_plant routes through Stage 1 detector + MaxLogit OOD gate, not RAPS'
cfg['cal_sources'] = ['plantdoc', 'inat']
Path('models/conformal.json').write_text(json.dumps(cfg, indent=2))
print('conformal:', cfg)

# 6. Persist class list + img_size — production reads exactly what we shipped.
classes_payload = {
    'model_version': '2.1',
    'class_names': CLASS_NAMES,
    'detector_classes': ['plant_leaf'],
    'training': {
        'img_size': IMG,
        'batch_size': BATCH,
        'loss': 'logit_adjustment_ce',
        'logit_adjustment_tau': LA_TAU,
        'label_smoothing': LABEL_SMOOTHING,
        'use_weighted_sampler': bool(USE_WEIGHTED_SAMPLER),
        'optimizer': 'AdamW',
        'weight_decay': 1e-4,
        'lr_schedule': 'cosine + warmup',
        'precision': 'bf16-mixed' if device == 'cuda' else 'fp32',
        'primary_metric': 'val_realworld_disease_macro_f1',
        'primary_val_set': 'data_v2/classification/val/realworld',
        'isolated_val_slice': 'data_v2/classification/val/plantvillage_studio',
    },
    'distillation': {
        'teacher': 'tf_efficientnetv2_s.in21k_ft_in1k',
        'student': 'mobilenet_v3_large',
        'loss': 'alpha*KD + (1-alpha)*CE_LA',
        'alpha': ALPHA,
        'tau':   T_KD,
        'ema_decay': EMA_DECAY,
        'epochs': KD_EPOCHS,
    },
    'ood': {
        'score': 'max_logit',
        'max_logit_floor': 4.0,
        '_score_note': 'MaxLogit (Hendrycks 2019) — robust under temperature scaling and label smoothing where MSP fails.',
    },
}
Path('models/classes.json').write_text(json.dumps(classes_payload, indent=2))
print(f'classes.json: {len(CLASS_NAMES)} classes')

# 7. Final metrics across all val slices via the deployed INT8 ORT session.
class _OnnxModelAdapter:
    """Thin wrapper so evaluate() can call model(x) on an ORT session."""
    def __init__(self, sess): self.sess = sess
    def eval(self): return self
    def __call__(self, x):
        out = _ort_logits(self.sess, x.detach().cpu().numpy().astype(np.float32))
        return torch.from_numpy(out).to(x.device)

int8_adapter = _OnnxModelAdapter(int8_sess)
ONNX_EVAL_DEVICE = 'cpu'
final_metrics = {
    'indomain':  evaluate(int8_adapter, val_indomain_dl,  CLASS_NAMES, ONNX_EVAL_DEVICE, 'final val_indomain'),
    'realworld': evaluate(int8_adapter, val_realworld_dl, CLASS_NAMES, ONNX_EVAL_DEVICE, 'final val_realworld'),
}
if val_studio_dl is not None:
    final_metrics['studio'] = evaluate(int8_adapter, val_studio_dl, CLASS_NAMES, ONNX_EVAL_DEVICE, 'final val_studio')
Path('models/training_metrics.json').write_text(json.dumps(final_metrics, indent=2))

# 8. Model card.
model_card = {
    'version': '2.1',
    'trained_at': datetime.datetime.now(datetime.timezone.utc).replace(microsecond=0).isoformat(),
    'classes': CLASS_NAMES,
    'img_size': IMG,
    'temperature': T_val,
    'conformal': cfg,
    'train_counts':           {c: count_images(TRAIN_ROOT  / c) for c in CLASS_NAMES},
    'val_realworld_counts':   {c: count_images(VAL_PRIMARY / c) for c in CLASS_NAMES},
    'val_indomain_counts':    {c: count_images(VAL_INDOMAIN/ c) for c in CLASS_NAMES},
    'temperature_cal_counts': {c: count_images(TEMP_CAL    / c) for c in CLASS_NAMES},
    'conformal_cal_counts':   {c: count_images(CONFORMAL_CAL/ c) for c in CLASS_NAMES},
    'val_studio_counts':      {c: count_images(VAL_STUDIO  / c) for c in CLASS_NAMES} if val_studio_dl is not None else None,
    'hyperparams': {
        'teacher': 'tf_efficientnetv2_s.in21k_ft_in1k',
        'student': 'mobilenet_v3_large',
        'kd_alpha': ALPHA, 'kd_tau': T_KD, 'la_tau': LA_TAU,
        'label_smoothing': LABEL_SMOOTHING,
        'use_weighted_sampler': bool(USE_WEIGHTED_SAMPLER),
        'batch': BATCH, 'img_size': IMG,
        'ema_decay': EMA_DECAY,
    },
    'int8': {
        'drift': drift, 'f1_fp32': f1_fp32, 'f1_int8': f1_int8,
        'warn_drift': WARN_DRIFT, 'fail_drift': FAIL_DRIFT,
        'activation_type': 'QUInt8', 'weight_type': 'QInt8',
        'per_channel': True, 'format': 'QDQ',
        'calibration_split': 'temperature_cal',
        'calibration_n': 500,
    },
    'kd': {'best_indomain_disease_f1': best_f1, 'best_epoch': best_ep},
    'reporting_backend': 'onnxruntime_int8_cpu',
    'reproducibility': {
        'seed': SEED,
        'torch': torch.__version__,
        'onnxruntime': ort.__version__,
        'cudnn_deterministic': bool(getattr(torch.backends.cudnn, 'deterministic', False)),
        'cudnn_benchmark':     bool(getattr(torch.backends.cudnn, 'benchmark', True)),
    },
}
Path('models/model_card.json').write_text(json.dumps(model_card, indent=2))
print('Persisted: training_metrics.json + model_card.json')

# 9. Ship gate — runs last so we never accidentally publish a broken model.
# - MIN_DISEASE_F1 / MAX_HEALTHY_BIAS / zero_recall: catch the obvious
#   failure modes from v2.0.
# - MIN_HEALTHY_RECALL: catches the inverse of healthy_bias — a model
#   that never predicts `healthy` and labels every plant as some
#   disease still passes disease_macro_f1, but ships false positives
#   in production. The bias check only catches over-prediction.
MIN_DISEASE_F1, MAX_HEALTHY_BIAS, MIN_HEALTHY_RECALL = 0.70, 1.50, 0.70
rw = final_metrics['realworld']
disease_classes = [c for c in CLASS_NAMES if c not in ('healthy', 'not_a_plant')]
zero_recall = [c for c in disease_classes if rw['per_class_recall'].get(c, 0.0) <= 0.0]
healthy_recall = float(rw['per_class_recall'].get('healthy', 0.0))
failures = []
if rw['disease_macro_f1'] < MIN_DISEASE_F1:
    failures.append(f'disease_macro_f1={rw["disease_macro_f1"]:.3f} < {MIN_DISEASE_F1}')
if rw['healthy_bias'] >= MAX_HEALTHY_BIAS:
    failures.append(f'healthy_bias={rw["healthy_bias"]:.2f} >= {MAX_HEALTHY_BIAS}')
if healthy_recall < MIN_HEALTHY_RECALL:
    failures.append(f'recall[healthy]={healthy_recall:.3f} < {MIN_HEALTHY_RECALL} (false-positive disease risk)')
if zero_recall:
    failures.append(f'zero recall classes: {zero_recall}')
if failures:
    raise RuntimeError('SHIP GATE FAILED: ' + '; '.join(failures))
print('SHIP GATE PASSED')

# 10. Detector artefact contract.
missing = [s for s in ('detector.pt', 'detector.onnx')
           if not Path(f'models/{s}').exists()]
if missing:
    print(f'WARN: {missing} not in models/ — drop the shipped '
          f'server/models/detector.{{pt,onnx}} pair into ./models/ '
          f'before assembling the deploy bundle, or re-run §3 with '
          f'a YOLO data.yaml under data_v2/.')

log_run({'phase': 'export_done', 'f1_fp32': f1_fp32, 'f1_int8': f1_int8, 'T': T_val,
         'q_hat': cfg['q_hat'], 'empirical_coverage': cfg['empirical_coverage']})


## §7 — Evaluation contract

| Slice | Role | Pass criterion |
|---|---|---|
| `val_indomain` | training-loop metric, drives early stopping | tracked; best checkpoint restored |
| `val_realworld` | **primary — ship/no-ship gate** | disease macro-F1 ≥ **0.70**; no disease class with recall = 0; healthy-bias < 1.5 |
| `val_studio` | PlantVillage transfer / background-bias monitor | tracked, never blocking |
| `temperature_cal` | INT8 activation calibration + temperature scaling | held out; never used for final reporting |
| `conformal_cal` | RAPS calibration | held out; target empirical coverage ≈ 1 − α |

Persisted artefacts (all written to `models/` in the notebook CWD):

* `classifier_int8.onnx`, `classifier.onnx`, `classifier.pth` — model weights.
* `classes.json` — the contract the server reads at startup.
* `temperature.json`, `conformal.json` — calibration configs fitted on the **deployed INT8 ONNX session**, not the FP32 torch model.
* `detector.{pt,onnx}` — Stage 1 weights (re-trained from §3 if a YOLO `data.yaml` is supplied; the shipped pair otherwise).
* `dinov2_probe.json` — diagnostic linear-probe curve.
* `training_metrics.json` — structured per-class recall + confusion matrix on every val slice.
* `model_card.json` — provenance: train/val counts per class, hyperparams, INT8 drift, best-KD epoch, reproducibility flags.
* `run_log.jsonl` — one JSON-line per epoch / phase, useful for post-mortem.
